# HW6 — Option 3: Predict Sentiment
### Will Kirk, April 2026
**Target Ticker: TSLA**

---
## Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install scipy==1.15.1

In [ ]:
import pandas as pd
import string
import matplotlib.pyplot as plt
import numpy as np
import re
import shap

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, MinMaxScaler
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, classification_report, accuracy_score

from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import xgboost as xgb

import boto3
import sagemaker
from sagemaker.predictor import Predictor
from sagemaker.serializers import NumpySerializer
from sagemaker.deserializers import NumpyDeserializer
from sagemaker.sklearn.model import SKLearnModel

from joblib import dump, load
import tarfile
import os

In [ ]:
import pkg_resources
installedPackages = {pkg.key for pkg in pkg_resources.working_set}
required = {'nltk', 'spacy', 'textblob', 'gensim'}
missing = required - installedPackages
if missing:
    !pip install nltk==3.9
    !pip install textblob==0.19.0
    !pip install gensim==4.4.0

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker_tab')

In [ ]:
import sys, os, importlib
module_path = os.path.abspath('..')
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import src.Custom_Classes
import src.feature_utils
importlib.reload(src.Custom_Classes)
importlib.reload(src.feature_utils)
from src.Custom_Classes import Word2VecTransformer

---
## Part 1 — Predicting Sentiment from Headlines
### 1.1 Load Labelled News Data

In [ ]:
dataset = pd.read_csv(r'LabelledNewsData_HW.csv', encoding='cp1252')
print(dataset.shape)
dataset.head()

### 1.2 Text Preprocessing (25 pts)
Steps applied to every headline:
1. Expand contractions (can't → cannot)
2. NER removal — locations, organisations, person names, dates, quantities, monetary values
3. Remove numbers and special characters
4. POS tagging — keep only Nouns and Verbs
5. Lemmatize remaining tokens

In [ ]:
# ── 1. Contractions ────────────────────────────────────────────────────────────
CONTRACTIONS = {
    "can't": "cannot", "won't": "will not", "n't": " not",
    "'re": " are", "'s": " is", "'d": " would", "'ll": " will",
    "'t": " not", "'ve": " have", "'m": " am"
}

def expand_contractions(text):
    for contraction, expansion in CONTRACTIONS.items():
        text = text.replace(contraction, expansion)
    return text

# ── 2. NER removal ─────────────────────────────────────────────────────────────
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.tree import Tree

NER_LABELS_TO_REMOVE = {
    'PERSON', 'ORGANIZATION', 'GPE',   # GPE = geo-political entity (locations)
    'LOCATION', 'FACILITY', 'DATE',
    'TIME', 'MONEY', 'PERCENT', 'QUANTITY'
}

def remove_named_entities(text):
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    chunked = ne_chunk(tagged, binary=False)
    result = []
    for subtree in chunked:
        if isinstance(subtree, Tree):
            if subtree.label() not in NER_LABELS_TO_REMOVE:
                result.extend([token for token, pos in subtree.leaves()])
        else:
            result.append(subtree[0])
    return " ".join(result)

# ── 3. Remove numbers & special characters ─────────────────────────────────────
def remove_numbers_and_special(text):
    text = re.sub(r'\$[\d,.]+', '', text)   # monetary like $20
    text = re.sub(r'\d+', '', text)          # remaining numbers
    text = re.sub(r'[^\w\s]', '', text)      # punctuation / special chars
    return text.strip()

# ── 4. POS tagging — keep Nouns (NN*) and Verbs (VB*) only ────────────────────
from nltk.corpus import stopwords

def keep_nouns_and_verbs(text):
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    kept   = [word for word, pos in tagged
               if pos.startswith('NN') or pos.startswith('VB')]
    return " ".join(kept)

# ── 5. Lemmatize ───────────────────────────────────────────────────────────────
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize(text):
    tokens = word_tokenize(text)
    return " ".join([lemmatizer.lemmatize(t) for t in tokens])

# ── Combined pipeline function ─────────────────────────────────────────────────
def full_preprocess(text):
    text = str(text).lower()
    text = expand_contractions(text)
    text = remove_named_entities(text)
    text = remove_numbers_and_special(text)
    text = keep_nouns_and_verbs(text)
    text = lemmatize(text)
    return text

print("Preprocessing functions defined.")

In [ ]:
# Apply preprocessing — use last 1000 rows as per template
subset_dataset = dataset.iloc[-1000:].copy()
subset_dataset['cleaned_headline'] = subset_dataset['headline'].apply(full_preprocess)

Y = subset_dataset['sentiment']
X = subset_dataset[['cleaned_headline']].rename(columns={'cleaned_headline': 'headline'})

subset_dataset[['headline', 'cleaned_headline', 'sentiment']].head(5)

### 1.3 Train / Test Split

In [ ]:
num_folds = 20
scoring   = 'f1_weighted'

validation_size = 0.2
seed = 7
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=validation_size, random_state=seed
)

### 1.4 Word2Vec Pipeline — Sklearn Preprocessing Steps

In [ ]:
def remove_stop_words(text_series):
    stop_list = set(ENGLISH_STOP_WORDS)
    def cleaner(text):
        return " ".join([w for w in str(text).split() if w.lower() not in stop_list])
    return text_series.iloc[:, 0].apply(cleaner).to_frame()

def remove_punctuation(text_series):
    punct_table = str.maketrans('', '', string.punctuation)
    def cleaner(text):
        return str(text).translate(punct_table)
    return text_series.iloc[:, 0].apply(cleaner).to_frame()

headline_pipe = Pipeline([
    ('stop_words',  FunctionTransformer(remove_stop_words)),
    ('punctuation', FunctionTransformer(remove_punctuation))
])

preprocessor = ColumnTransformer(
    transformers=[('headline_process', headline_pipe, ['headline'])],
    remainder='passthrough'
)

pipeline_steps = [
    ('preprocessing',         preprocessor),
    ('feature_representation', Word2VecTransformer(vector_size=100)),
    ('model',                  LogisticRegression())   # placeholder, replaced in loop below
]

### 1.5 Evaluate 6 Classifiers (10 pts)

In [ ]:
models = [
    ('LogisticRegression', LogisticRegression(max_iter=300)),
    ('KNN',               KNeighborsClassifier()),
    ('DecisionTree',      DecisionTreeClassifier()),
    ('SVC',               SVC(probability=True)),
    ('MLP',               MLPClassifier(max_iter=300)),
    ('RandomForest',      RandomForestClassifier()),
]

names          = []
kfold_results  = []
test_results   = []
train_results  = []
fitted_models  = {}   # store fitted pipelines so we can pick the best

for name, model in models:
    names.append(name)
    steps = pipeline_steps[:-1] + [('model', model)]
    clf_pipeline = Pipeline(steps)

    kfold      = KFold(n_splits=num_folds, shuffle=False)
    cv_results = cross_val_score(
        estimator=clf_pipeline, X=X_train, y=Y_train,
        scoring=scoring, cv=kfold
    )
    kfold_results.append(cv_results)

    clf_pipeline.fit(X_train, Y_train)
    fitted_models[name] = clf_pipeline

    train_result = f1_score(Y_train, clf_pipeline.predict(X_train), average='weighted')
    test_result  = f1_score(Y_test,  clf_pipeline.predict(X_test),  average='weighted')
    train_results.append(train_result)
    test_results.append(test_result)

    print("%s: cv=%.4f (±%.4f)  train=%.4f  test=%.4f" % (
        name, cv_results.mean(), cv_results.std(), train_result, test_result
    ))

In [ ]:
import matplotlib.pyplot as pyplot

fig = pyplot.figure(figsize=(15, 6))
fig.suptitle('Algorithm Comparison — K-Fold CV (f1_weighted)')
ax = fig.add_subplot(111)
pyplot.boxplot(kfold_results)
ax.set_xticklabels(names, rotation=15)
pyplot.show()

In [ ]:
ind   = np.arange(len(names))
width = 0.35

fig = pyplot.figure(figsize=(15, 6))
fig.suptitle('Train vs Test F1 Score')
ax  = fig.add_subplot(111)
pyplot.bar(ind - width/2, train_results, width=width, label='Train')
pyplot.bar(ind + width/2, test_results,  width=width, label='Test')
pyplot.legend()
ax.set_xticks(ind)
ax.set_xticklabels(names, rotation=15)
pyplot.show()

### 1.6 Choose Best Classifier (10 pts)

In [ ]:
# Pick the classifier with the highest test F1 score
best_name     = names[int(np.argmax(test_results))]
best_clf      = fitted_models[best_name]
print(f"Best classifier: {best_name}  (test F1 = {max(test_results):.4f})")

### 1.7 Apply Best Classifier to DataWithSentimentsResults_HW.csv (20 pts)

In [ ]:
sent_dataset = pd.read_csv(r'DataWithSentimentsResults_HW.csv', sep='|')
print("Tickers:", pd.unique(sent_dataset.ticker).tolist())
print("Date range:", sent_dataset.date.min(), "→", sent_dataset.date.max())
sent_dataset.head()

In [ ]:
# Preprocess headlines before predicting
sent_dataset['cleaned_headline'] = sent_dataset['headline'].apply(full_preprocess)
X_news = sent_dataset[['cleaned_headline']].rename(columns={'cleaned_headline': 'headline'})

# predict_proba[:, 1] → probability of positive sentiment
sent_dataset['PredictedSentiment'] = best_clf.predict_proba(X_news)[:, 1]
sent_dataset[['ticker', 'date', 'headline', 'PredictedSentiment']].head()

### 1.8 Correlation Between PredictedSentiment and Stock Returns (10 pts)

In [ ]:
stock_dataset = pd.read_csv(r'stock_dataset_2010_2018.csv')
stock_dataset['date'] = pd.to_datetime(stock_dataset['Date'])

# Calculate daily return for all tickers in the sentiment dataset
tickers_in_sent = pd.unique(sent_dataset.ticker).tolist()
corr_results = {}

for t in tickers_in_sent:
    if t not in stock_dataset.columns:
        continue
    tmp = stock_dataset[['date', t]].copy()
    tmp['return'] = tmp[t].pct_change()
    tmp_sent = (
        sent_dataset[sent_dataset.ticker == t][['date', 'PredictedSentiment']]
        .copy()
    )
    tmp_sent['date'] = pd.to_datetime(tmp_sent['date'])
    tmp_sent = tmp_sent.groupby('date').mean().reset_index()
    merged = pd.merge(tmp, tmp_sent, on='date', how='inner').dropna()
    if len(merged) > 10:
        corr_results[t] = merged['PredictedSentiment'].corr(merged['return'])

corr_series = pd.Series(corr_results).sort_values(ascending=False)
print(corr_series)
print(f"\nMean absolute correlation: {corr_series.abs().mean():.4f}")
print("Compare to: lexicon=0.16, LSTM=0.10, TextBlob=0.05")

---
## Part 2 — Predicting TSLA Signal (BUY / HOLD / SELL)
### 2.1 Feature Engineering — Target Ticker: TSLA

In [ ]:
ticker = 'TSLA'

# Other tickers' PredictedSentiment (pivoted by date)
features = sent_dataset[sent_dataset.ticker != ticker].copy()
features  = features[['ticker', 'date', 'PredictedSentiment']]
features['date'] = pd.to_datetime(features['date'])
features  = features.groupby(['ticker', 'date']).mean().reset_index()
features  = features.pivot(index='date', columns='ticker', values='PredictedSentiment').reset_index()
features.head()

In [ ]:
# TSLA's own PredictedSentiment
sent_dataset_target = sent_dataset[sent_dataset.ticker == ticker].copy()
sent_dataset_target  = sent_dataset_target[['date', 'PredictedSentiment']]
sent_dataset_target['date'] = pd.to_datetime(sent_dataset_target['date'])
sent_dataset_target  = sent_dataset_target.groupby('date').mean().reset_index()
sent_dataset_target.head()

In [ ]:
# Load TSLA price data
stock_df = pd.read_csv(r'stock_dataset_2010_2018.csv')
stock_df  = stock_df[[ticker, 'Date']].copy()
stock_df.rename(columns={'Date': 'date', ticker: 'Close'}, inplace=True)
stock_df['date'] = pd.to_datetime(stock_df['date'])
stock_df.head()

In [ ]:
# Merge price + TSLA sentiment
dataset = pd.merge(stock_df, sent_dataset_target, on='date', how='left')
dataset = dataset.fillna(method='ffill')

# Merge other-ticker sentiments
dataset = pd.merge(dataset, features, on='date', how='left')
dataset = dataset.fillna(method='ffill')
dataset.tail()

### 2.2 Create Signal Target Variable

In [ ]:
dataset['Next_Day_Return'] = dataset['Close'].pct_change().shift(-1)

threshold  = 0.005
conditions = [
    (dataset['Next_Day_Return'] >  threshold),
    (dataset['Next_Day_Return'] < -threshold)
]
choices = [2, 0]   # 2 = BUY, 0 = SELL, default 1 = HOLD

dataset['signal'] = np.select(conditions, choices, default=1)
dataset.dropna(inplace=True)

print(dataset['signal'].value_counts())
dataset.head()

### 2.3 Correlation Analysis — Pick Best Features

In [ ]:
# Review correlations to choose which columns to include in X
numeric_cols = dataset.select_dtypes(include=np.number).columns.tolist()
corr_with_signal = dataset[numeric_cols].corr()['signal'].abs().sort_values(ascending=False)
print(corr_with_signal)

# ── UPDATE THIS LIST after reviewing the output above ────────────────────────
# Use the top 3 other-ticker columns + TSLA's PredictedSentiment
# The template uses ['ADBE','AMZN','WMT','PredictedSentiment'] for NFLX
# For TSLA, swap in whichever columns rank highest in the correlation output
FEATURE_COLS = ['ADBE', 'AMZN', 'WMT', 'PredictedSentiment']
# ─────────────────────────────────────────────────────────────────────────────
print("\nUsing features:", FEATURE_COLS)

### 2.4 Train / Test Split

In [ ]:
subset_dataset = dataset.iloc[-1000:]
Y = subset_dataset['signal']
X = subset_dataset[FEATURE_COLS]

validation_size = 0.2
train_size = int(len(X) * (1 - validation_size))
X_train, X_test = X[0:train_size], X[train_size:]
Y_train, Y_test = Y[0:train_size], Y[train_size:]

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

### 2.5 Define Pipeline

In [ ]:
num_folds = 10
seed      = 7
scoring   = 'f1_weighted'

pipeline_steps = [
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler()),
    ('sampler', SMOTE(random_state=42, sampling_strategy='auto')),
    ('model',   xgb.XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        n_estimators=100,
        learning_rate=0.1
    ))
]

### 2.6 Grid Search & Model Tuning

In [ ]:
classification_pipeline = Pipeline(pipeline_steps)

param_grid = [{
    'imputer__strategy': ['mean', 'median'],
    'scaler':            [StandardScaler(), MinMaxScaler()],
    'sampler':           [SMOTE(random_state=42),
                          RandomUnderSampler(random_state=42),
                          BorderlineSMOTE(random_state=42)],
    'model__learning_rate': [0.01, 0.1, 0.2],
}]

kfold       = KFold(n_splits=5, shuffle=False)
grid_search = GridSearchCV(
    estimator  = classification_pipeline,
    param_grid = param_grid,
    cv         = kfold,
    scoring    = scoring
)
grid_search.fit(X_train, Y_train)

print("Best params:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

### 2.7 Save Best Model (5 pts)

In [ ]:
best_pipeline = grid_search.best_estimator_

# Evaluate on test set
y_pred = best_pipeline.predict(X_test)
HW6_Accuracy = accuracy_score(Y_test, y_pred)
print(f"HW6 Test Accuracy: {HW6_Accuracy:.4f}")
print(classification_report(Y_test, y_pred, target_names=['SELL', 'HOLD', 'BUY']))

# Save
filename = r'./finalized_sentiment_model.joblib'
dump(best_pipeline, filename)
print(f"Model saved to {filename}")

# Verify round-trip
best_pipeline_loaded = load(filename)
print("Reload accuracy:", accuracy_score(Y_test, best_pipeline_loaded.predict(X_test)))

In [ ]:
with tarfile.open('finalized_sentiment_model.tar.gz', 'w:gz') as tar:
    tar.add(r'./finalized_sentiment_model.joblib', arcname='finalized_sentiment_model.joblib')
    tar.add(r'../src', arcname='src')
print("tar.gz created.")

### 2.8 Feature Importance (5 pts)

In [ ]:
feature_names = best_pipeline[:-1].get_feature_names_out()
importances   = best_pipeline.named_steps['model'].feature_importances_

Importance = pd.DataFrame(
    {'Importance': np.abs(importances) * 100},
    index=feature_names
)
Importance.sort_values('Importance', ascending=True).plot(
    kind='barh', color='steelblue', figsize=(10, 5),
    title='Feature Importance — TSLA Signal Prediction'
)
plt.xlabel('Relative Importance (%)')
plt.tight_layout()
plt.show()

### 2.9 SHAP Explainability (5 pts)

In [ ]:
model = best_pipeline.named_steps['model']

# steps: imputer → scaler → sampler → model
# [:-2] keeps imputer + scaler only (no sampler, no model)
preprocessing_pipeline = Pipeline(steps=best_pipeline.steps[:-2])
X_train_transformed     = preprocessing_pipeline.transform(X_train)

explainer = shap.Explainer(model, X_train_transformed)

# Save explainer
with open('explainer_sentiment.shap', 'wb') as f:
    dump(explainer, f)
print("SHAP explainer saved.")

In [ ]:
# Reload and plot
with open('explainer_sentiment.shap', 'rb') as f:
    explainer = load(f)

X_test_transformed = preprocessing_pipeline.transform(X_test)
feature_names_out  = best_pipeline[:-2].get_feature_names_out()
X_test_transformed = pd.DataFrame(X_test_transformed, columns=feature_names_out)

shap_values = explainer(X_test_transformed)

# Waterfall for first test sample, class 0 (SELL)
shap.plots.waterfall(shap_values[0, :, 0])

---
## Part 3 — Deploy to AWS SageMaker (10 pts)

In [ ]:
session          = boto3.Session()
s3_client        = session.client('s3')
bucket_name      = 'will-kirk-s3-bucket'
sagemaker_session = sagemaker.Session(boto_session=session, default_bucket=bucket_name)

credentials           = session.get_credentials()
current_access_key    = credentials.access_key
current_secret_key    = credentials.secret_key
current_session_token = credentials.get_frozen_credentials().token

print(f"Access Key:    {current_access_key}")
print(f"Secret Key:    {current_secret_key}")
print(f"Session Token: {current_session_token}")

In [ ]:
# Clear bucket before uploading
s3_resource = boto3.resource('s3')
s3_bucket   = s3_resource.Bucket(bucket_name)
s3_bucket.objects.all().delete()
print("Bucket cleared.")

In [ ]:
# Upload SHAP explainer
s3_client.upload_file(
    Filename = './explainer_sentiment.shap',
    Bucket   = bucket_name,
    Key      = 'explainer/explainer_sentiment.shap'
)
print("Explainer uploaded.")

In [ ]:
# Upload model tar.gz
s3_path_key = 'sklearn-pipeline-deployment'
filename    = 'finalized_sentiment_model.tar.gz'
s3_client.upload_file(
    Filename = filename,
    Bucket   = bucket_name,
    Key      = f"{s3_path_key}/{os.path.basename(filename)}"
)
model_s3_uri = f"s3://{bucket_name}/{s3_path_key}/{filename}"
print(f"Model uploaded: {model_s3_uri}")

In [ ]:
# Write requirements.txt
with open('requirements.txt', 'w') as f:
    f.write('numpy==1.26.4\n')
    f.write('scipy==1.12.0\n')
    f.write('scikit-learn==1.3.2\n')
    f.write('statsmodels==0.14.1\n')
    f.write('pandas==2.2.0\n')
    f.write('xgboost\n')
    f.write('nltk==3.9\n')
    f.write('textblob==0.19.0\n')
    f.write('gensim==4.4.0\n')
    f.write('imbalanced-learn==0.12.0\n')
print("requirements.txt written.")

In [ ]:
model_name    = 'TSLA-Sentiment-Signal-Model'
endpoint_name = 'tsla-signal-endpoint-1'   # ← paste this into secrets.toml AWS_ENDPOINT
instance_type = 'ml.m5.large'
custom_code_uri = f"s3://{bucket_name}/customCode/"

sklearn_model = SKLearnModel(
    model_data      = model_s3_uri,
    role            = sagemaker.get_execution_role(),
    entry_point     = 'inference_sentiment.py',
    framework_version = '1.2-1',
    py_version      = 'py3',
    dependencies    = ['requirements.txt'],
    source_dir      = '.',
    name            = model_name,
    sagemaker_session = sagemaker_session,
    code_location   = custom_code_uri
)

In [ ]:
print(f"Deploying {model_name} → {endpoint_name} …")
predictor = sklearn_model.deploy(
    initial_instance_count = 1,
    instance_type          = instance_type,
    endpoint_name          = endpoint_name,
)
print(f"\nDeployment complete! Endpoint: {endpoint_name}")

In [ ]:
# Quick smoke-test against the live endpoint
from sagemaker.serializers import NumpySerializer
from sagemaker.deserializers import NumpyDeserializer

predictor.serializer   = NumpySerializer()
predictor.deserializer = NumpyDeserializer()

y_pred_endpoint = predictor.predict(X_test.values.astype(np.float32))
HW6_Accuracy_endpoint = accuracy_score(Y_test, y_pred_endpoint)
print(f"Endpoint Accuracy: {HW6_Accuracy_endpoint:.4f}")

---
## Part 4 — Streamlit App (10 pts)

The Streamlit app file is `StreamlitApp_HW6_TSLA.py`.

**To run locally:**
```bash
streamlit run StreamlitApp_HW6_TSLA.py
```

**To deploy on Streamlit Cloud:**
1. Push the repo to GitHub
2. Go to [share.streamlit.io](https://share.streamlit.io) → New app
3. Set the main file to `StreamlitApp_HW6_TSLA.py`
4. In **Settings → Secrets**, paste your `.streamlit/secrets.toml` content

**Input features the app expects** (must match `FEATURE_COLS` above):

| Field | Description |
|---|---|
| `ADBE` | PredictedSentiment for Adobe |
| `AMZN` | PredictedSentiment for Amazon |
| `WMT` | PredictedSentiment for Walmart |
| `PredictedSentiment` | TSLA's own PredictedSentiment |

> ⚠️ If your correlation analysis in cell 2.3 shows different top columns, update `FEATURE_COLS` **and** the `MODEL_KEYS` / `inputs` list in `StreamlitApp_HW6_TSLA.py` to match.